# 04 · Machine Learning Baseline — OASIS-2

## Pregunta predictiva

**¿Podemos utilizar la información disponible en la primera visita para distinguir a los pacientes que posteriormente desarrollarán demencia (`Converted`) de aquellos que permanecen `Nondemented`?**

Este análisis utiliza únicamente la visita inicial. Los sujetos que ya pertenecen al grupo `Demented` se excluyen porque la pregunta no es detectar una demencia ya presente, sino estudiar si existe alguna señal basal asociada a la conversión posterior.

> **Importante:** este proyecto es exploratorio y no desarrolla una herramienta diagnóstica. OASIS-2 tampoco permite demostrar una predicción presintomática de Alzheimer a cinco años.

> **Datos:** el dataset OASIS-2 no se distribuye en este repositorio. Se conservan código, métricas agregadas e interpretaciones, evitando publicar registros individuales.


## Conceptos de Machine Learning utilizados

Este notebook está pensado para que pueda seguirlo también una persona sin experiencia previa en Machine Learning.

| Concepto | Explicación |
|---|---|
| **Variable objetivo (`y`)** | Lo que queremos clasificar: `0 = Nondemented` y `1 = Converted`. |
| **Variables predictoras (`X`)** | Información disponible en la primera visita: sexo, edad, educación, SES y determinadas medidas cognitivas/MRI. |
| **StandardScaler** | Estandariza las variables numéricas para que estén en escalas comparables. Es especialmente importante para modelos como regresión logística y SVM. |
| **OneHotEncoder** | Convierte una variable categórica como sexo en variables numéricas que el modelo puede utilizar. |
| **Pipeline** | Encadena preprocesamiento y modelo para que las transformaciones se aprendan únicamente con los datos de entrenamiento de cada partición, evitando *data leakage*. |
| **Class weight = balanced** | Da mayor peso a la clase minoritaria (`Converted`) para que el modelo no pueda obtener un buen resultado limitándose a predecir casi siempre la clase mayoritaria. |
| **Validación cruzada estratificada** | Divide repetidamente la muestra manteniendo aproximadamente la proporción de Nondemented/Converted en cada partición. |
| **Ablación de variables** | Repetir el experimento eliminando determinadas variables para comprobar cuánto depende el rendimiento de ellas. |


## ¿Cómo evaluaremos los modelos?

Con una cohorte desbalanceada, **accuracy por sí sola puede ser engañosa**. Si casi todos los pacientes son Nondemented, un modelo puede acertar muchas veces simplemente prediciendo siempre la clase mayoritaria.

Por ello se utilizan varias métricas:

- **Recall (sensibilidad):** de todos los Converted reales, qué proporción consigue detectar el modelo.
- **Precision (valor predictivo positivo):** de todos los pacientes que el modelo etiqueta como Converted, qué proporción realmente pertenece a ese grupo.
- **F1-score:** combina precision y recall.
- **ROC-AUC:** mide la capacidad global del modelo para ordenar/separar ambas clases a distintos umbrales.
- **PR-AUC:** resume la relación Precision–Recall y resulta especialmente informativa cuando la clase positiva es poco frecuente.

Ninguna métrica debe interpretarse de forma aislada.


## 1. Carga de datos y selección de la cohorte baseline


In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [2]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

In [4]:
df_oasis2 = pd.read_excel("../data/oasis_2.xlsx")

In [7]:
df_selection = df_oasis2[df_oasis2['Visit']==1]

In [9]:
df_baselineSelection2 = df_selection[
    (df_selection['Group'] == 'Nondemented') |
    (df_selection['Group'] == 'Converted')
]

In [11]:
df_baselineSelection2.shape

(86, 15)

In [13]:
df_baselineSelection2['Group'].value_counts()

Group
Nondemented    72
Converted      14
Name: count, dtype: int64

La cohorte predictiva queda formada por **86 pacientes: 72 Nondemented y 14 Converted**. Esto supone un desbalance importante: aproximadamente el 84 % pertenece a la clase Nondemented.

Este punto condicionará toda la evaluación posterior y explica por qué una accuracy cercana al 84 % podría no representar un modelo útil.


## 2. Variables predictoras y preprocesamiento

En el primer experimento se utilizan `M/F`, `Age`, `EDUC`, `SES`, `MMSE`, `CDR`, `eTIV` y `nWBV`.

Se excluyen identificadores (`Subject ID`, `MRI ID`), variables temporales de seguimiento (`Visit`, `MR Delay`) y `ASF`, ya que el EDA mostró que es prácticamente redundante con `eTIV`.

La variable objetivo se codifica como `0 = Nondemented` y `1 = Converted`.


In [15]:
y = df_baselineSelection2['Group'].map({
    'Nondemented': 0,
    'Converted':1
})

In [17]:
X = df_baselineSelection2[['M/F','Age','EDUC','SES','MMSE','CDR','eTIV','nWBV']]

In [19]:
X.isnull().sum()

M/F     0
Age     0
EDUC    0
SES     0
MMSE    0
CDR     0
eTIV    0
nWBV    0
dtype: int64

In [21]:
numeric_features = ['Age','EDUC','SES','MMSE','CDR','eTIV','nWBV']

In [23]:
categorical_features = ['M/F']

In [25]:
scaler = StandardScaler()

In [27]:
encoder = OneHotEncoder()

In [29]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler,numeric_features),
        ('cat', encoder,categorical_features)
    ]
)

## 3. Estrategia de validación

Se utiliza **Repeated Stratified 5-Fold Cross-Validation**, con 5 particiones repetidas 10 veces (50 evaluaciones).

La estratificación intenta mantener la proporción de ambas clases en cada fold. Las repeticiones permiten observar cuánto cambia el rendimiento al modificar las particiones, algo especialmente importante con solo 14 sujetos Converted.

Las 50 evaluaciones **no representan 50 muestras independientes**: son diferentes particiones de los mismos 86 pacientes.


In [31]:
cv_repeated = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

## 4. Regresión logística: efecto del desbalance

La regresión logística proporciona un primer modelo lineal e interpretable. Se compara su versión estándar con una versión `class_weight='balanced'`.


In [33]:
#primer modelo
logistic_model = LogisticRegression()

In [35]:
#primer modelo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', logistic_model )
])

In [37]:
#segundo modelo
logistic_model_balanced = LogisticRegression(
    class_weight='balanced'
)

In [39]:
pipeline_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', logistic_model_balanced)
])

In [41]:
results_repeated = cross_validate(
    pipeline,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [43]:
results_repeated_balanced = cross_validate(
    pipeline_balanced,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

In [45]:
metricas = [
    'accuracy',
    'recall',
    'precision',
    'f1',
    'roc_auc',
    'pr_auc'
]

In [47]:
for metrica in metricas:
    print(metrica)
    print(results_repeated[f'test_{metrica}'].mean())
    print(results_repeated[f'test_{metrica}'].std())

accuracy
0.8314379084967319
0.03107215939131333
recall
0.006666666666666666
0.04666666666666666
precision
0.01
0.06999999999999999
f1
0.008
0.056
roc_auc
0.5671111111111112
0.20174647883756489
pr_auc
0.34977952330893514
0.17080695322070363


In [49]:
for metrica in metricas:
    print(metrica)
    print(results_repeated_balanced[f'test_{metrica}'].mean())
    print(results_repeated_balanced[f'test_{metrica}'].std())

accuracy
0.6020261437908497
0.10925455866268026
recall
0.4766666666666666
0.3091385881229755
precision
0.1892734487734488
0.119291228209974
f1
0.2644614274614275
0.16315110826029197
roc_auc
0.5396190476190477
0.1988518213725731
pr_auc
0.32974766736531436
0.16357576115382474


In [51]:
metricas = {
    'Accuracy': 'accuracy',
    'Recall': 'recall',
    'Precision': 'precision',
    'F1-score': 'f1',
    'ROC-AUC': 'roc_auc',
    'PR-AUC': 'pr_auc'
}

tabla_resultados = []

for nombre, metrica in metricas.items():

    media_normal = results_repeated[f'test_{metrica}'].mean()
    std_normal = results_repeated[f'test_{metrica}'].std()

    media_balanced = results_repeated_balanced[f'test_{metrica}'].mean()
    std_balanced = results_repeated_balanced[f'test_{metrica}'].std()

    tabla_resultados.append({
        'Metric': nombre,
        'Logistic Regression': f'{media_normal:.3f} ± {std_normal:.3f}',
        'Logistic Regression Balanced': f'{media_balanced:.3f} ± {std_balanced:.3f}'
    })

df_resultados = pd.DataFrame(tabla_resultados)

df_resultados

,Metric,Logistic Regression,Logistic Regression Balanced
0,Accuracy,0.831 ± 0.031,0.602 ± 0.109
1,Recall,0.007 ± 0.047,0.477 ± 0.309
2,Precision,0.010 ± 0.070,0.189 ± 0.119
3,F1-score,0.008 ± 0.056,0.264 ± 0.163
4,ROC-AUC,0.567 ± 0.202,0.540 ± 0.199
5,PR-AUC,0.350 ± 0.171,0.330 ± 0.164


### Interpretación

La regresión logística estándar obtiene una accuracy de aproximadamente **83 %**, pero su recall para Converted es prácticamente **0**. El resultado aparentemente bueno se debe a que el modelo aprende principalmente a predecir la clase mayoritaria.

Al aplicar balanceo, el recall aumenta hasta aproximadamente **48 %**, pero la accuracy disminuye y la precision continúa siendo baja. El modelo detecta más Converted a costa de generar más falsos positivos.

Este experimento muestra por qué **accuracy no es suficiente** en un problema desbalanceado.


## 5. Comparación de modelos clásicos

Se comparan tres familias de modelos:

- **Logistic Regression:** modelo lineal probabilístico e interpretable.
- **Random Forest:** conjunto de árboles de decisión capaz de modelar relaciones no lineales.
- **SVM lineal:** busca una frontera que maximice la separación entre las clases.

Para cada familia se evalúa una configuración estándar y otra balanceada. No se realiza ajuste exhaustivo de hiperparámetros: el objetivo es comparar enfoques clásicos de forma exploratoria, no optimizar una cifra de rendimiento.


In [53]:
preprocessor_rf = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('cat', encoder, categorical_features)
    ]
)

In [55]:
random_forest = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

In [57]:
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor_rf),
    ('classifier', random_forest)
])

In [59]:
results_rf = cross_validate(
    pipeline_rf,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [61]:
random_forest_balanced = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42
)

In [63]:
pipeline_rf_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor_rf),
    ('classifier', random_forest_balanced)
])

In [65]:
results_rf_balanced = cross_validate(
    pipeline_rf_balanced,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [67]:
import pandas as pd

metricas = {
    'Accuracy': 'accuracy',
    'Recall': 'recall',
    'Precision': 'precision',
    'F1-score': 'f1',
    'ROC-AUC': 'roc_auc',
    'PR-AUC': 'pr_auc'
}

tabla_resultados = []

for nombre, metrica in metricas.items():

    tabla_resultados.append({
        'Metric': nombre,

        'Logistic Regression':
            f"{results_repeated[f'test_{metrica}'].mean():.3f} ± "
            f"{results_repeated[f'test_{metrica}'].std():.3f}",

        'Logistic Balanced':
            f"{results_repeated_balanced[f'test_{metrica}'].mean():.3f} ± "
            f"{results_repeated_balanced[f'test_{metrica}'].std():.3f}",

        'Random Forest':
            f"{results_rf[f'test_{metrica}'].mean():.3f} ± "
            f"{results_rf[f'test_{metrica}'].std():.3f}",

        'Random Forest Balanced':
            f"{results_rf_balanced[f'test_{metrica}'].mean():.3f} ± "
            f"{results_rf_balanced[f'test_{metrica}'].std():.3f}"
    })

df_resultados = pd.DataFrame(tabla_resultados)

df_resultados

,Metric,Logistic Regression,Logistic Balanced,Random Forest,Random Forest Balanced
0,Accuracy,0.831 ± 0.031,0.602 ± 0.109,0.814 ± 0.045,0.822 ± 0.042
1,Recall,0.007 ± 0.047,0.477 ± 0.309,0.007 ± 0.047,0.007 ± 0.047
2,Precision,0.010 ± 0.070,0.189 ± 0.119,0.007 ± 0.047,0.020 ± 0.140
3,F1-score,0.008 ± 0.056,0.264 ± 0.163,0.007 ± 0.047,0.010 ± 0.070
4,ROC-AUC,0.567 ± 0.202,0.540 ± 0.199,0.466 ± 0.150,0.458 ± 0.157
5,PR-AUC,0.350 ± 0.171,0.330 ± 0.164,0.257 ± 0.130,0.262 ± 0.130


In [71]:
svm_model = SVC(
    kernel='linear',
    probability=True,
    random_state=42
)

In [73]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler,numeric_features),
        ('cat', encoder,categorical_features)
    ]
)

In [75]:
pipeline_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_model)
])

In [77]:
results_svm = cross_validate(
    pipeline_svm,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [80]:
svm_model_balanced = SVC(
    kernel='linear',
    class_weight='balanced',
    probability=True,
    random_state=42
)

In [82]:
pipeline_svm_balanced = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', svm_model_balanced)
])

In [84]:
results_svm_balanced = cross_validate(
    pipeline_svm_balanced,
    X,
    y,
    cv=cv_repeated,
    scoring={
        'accuracy': 'accuracy',
        'recall': 'recall',
        'precision': 'precision',
        'f1': 'f1',
        'roc_auc': 'roc_auc',
        'pr_auc': 'average_precision'
    }
)

In [85]:
tabla_resultados = []

for nombre, metrica in metricas.items():

    tabla_resultados.append({
        'Metric': nombre,

        'Logistic Regression':
            f"{results_repeated[f'test_{metrica}'].mean():.3f} ± "
            f"{results_repeated[f'test_{metrica}'].std():.3f}",

        'Logistic Balanced':
            f"{results_repeated_balanced[f'test_{metrica}'].mean():.3f} ± "
            f"{results_repeated_balanced[f'test_{metrica}'].std():.3f}",

        'Random Forest':
            f"{results_rf[f'test_{metrica}'].mean():.3f} ± "
            f"{results_rf[f'test_{metrica}'].std():.3f}",

        'Random Forest Balanced':
            f"{results_rf_balanced[f'test_{metrica}'].mean():.3f} ± "
            f"{results_rf_balanced[f'test_{metrica}'].std():.3f}",

        'SVM':
            f"{results_svm[f'test_{metrica}'].mean():.3f} ± "
            f"{results_svm[f'test_{metrica}'].std():.3f}",

        'SVM Balanced':
            f"{results_svm_balanced[f'test_{metrica}'].mean():.3f} ± "
            f"{results_svm_balanced[f'test_{metrica}'].std():.3f}"

    })

df_resultados = pd.DataFrame(tabla_resultados)

df_resultados

,Metric,Logistic Regression,Logistic Balanced,Random Forest,Random Forest Balanced,SVM,SVM Balanced
0,Accuracy,0.831 ± 0.031,0.602 ± 0.109,0.814 ± 0.045,0.822 ± 0.042,0.837 ± 0.023,0.597 ± 0.114
1,Recall,0.007 ± 0.047,0.477 ± 0.309,0.007 ± 0.047,0.007 ± 0.047,0.000 ± 0.000,0.493 ± 0.332
2,Precision,0.010 ± 0.070,0.189 ± 0.119,0.007 ± 0.047,0.020 ± 0.140,0.000 ± 0.000,0.196 ± 0.127
3,F1-score,0.008 ± 0.056,0.264 ± 0.163,0.007 ± 0.047,0.010 ± 0.070,0.000 ± 0.000,0.273 ± 0.172
4,ROC-AUC,0.567 ± 0.202,0.540 ± 0.199,0.466 ± 0.150,0.458 ± 0.157,0.572 ± 0.183,0.572 ± 0.212
5,PR-AUC,0.350 ± 0.171,0.330 ± 0.164,0.257 ± 0.130,0.262 ± 0.130,0.337 ± 0.160,0.362 ± 0.182


### Resultado general

Los modelos no balanceados presentan una accuracy elevada pero detectan muy pocos Converted. El balanceo mejora el recall en los modelos lineales, especialmente Logistic Regression y SVM, aunque con una pérdida importante de accuracy y una precision baja.

Random Forest no mostró una mejora relevante en la detección de Converted con estas variables y esta muestra.

En conjunto, la discriminación entre Nondemented y futuros Converted es **limitada e inestable**.


## 6. Ablación 1 — ¿Qué ocurre si eliminamos CDR?

`CDR` está muy próximo a la valoración clínica de demencia. Aunque en baseline casi todos los Converted tienen CDR = 0, conviene comprobar si el rendimiento depende de una variable tan relacionada con la clasificación clínica.

Se repiten exactamente los mismos modelos y la misma estrategia de validación eliminando CDR.


In [88]:
X_no_CDR = df_baselineSelection2[
    ['M/F', 'Age', 'EDUC', 'SES', 'MMSE', 'eTIV', 'nWBV']
]

In [90]:
numeric_features_no_CDR = [
    'Age',
    'EDUC',
    'SES',
    'MMSE',
    'eTIV',
    'nWBV'
]

In [92]:
preprocessor_no_CDR = ColumnTransformer(
    transformers=[
        ('num', scaler, numeric_features_no_CDR),
        ('cat', encoder, categorical_features)
    ]
)

In [94]:
preprocessor_rf_no_CDR = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features_no_CDR),
        ('cat', encoder, categorical_features)
    ]
)

In [96]:
df_baselineSelection2.groupby('Group')['CDR'].value_counts()

Group        CDR
Converted    0.0    13
             0.5     1
Nondemented  0.0    72
Name: count, dtype: int64

In [98]:
pd.crosstab(
    df_baselineSelection2['Group'],
    df_baselineSelection2['CDR']
)

CDR,0.0,0.5
Group,,
Converted,13,1
Nondemented,72,0


In [100]:
pipeline_logistic_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR),
    ('classifier', logistic_model)
])

In [102]:
pipeline_logistic_balanced_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR),
    ('classifier', logistic_model_balanced)
])

In [104]:
pipeline_svm_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR),
    ('classifier', svm_model)
])

In [106]:
pipeline_svm_balanced_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR),
    ('classifier', svm_model_balanced)
])

In [108]:
pipeline_random_forest_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_rf_no_CDR),
    ('classifier', random_forest)
])

In [110]:
pipeline_random_forest_balanced_no_CDR = Pipeline(steps=[
    ('preprocessor', preprocessor_rf_no_CDR),
    ('classifier', random_forest_balanced)
])

In [112]:
modelos_ML = {
    'Logistic': pipeline_logistic_no_CDR,
    'Logistic Balanced': pipeline_logistic_balanced_no_CDR,
    'Random Forest': pipeline_random_forest_no_CDR,
    'Random Forest Balanced': pipeline_random_forest_balanced_no_CDR,
    'SVM': pipeline_svm_no_CDR,
    'SVM balanced': pipeline_svm_balanced_no_CDR
}

In [114]:
resultados_no_CDR = {}

for nombre, modelo in modelos_ML.items():
    resultados_no_CDR[nombre] = cross_validate(
        modelo,
        X_no_CDR,
        y,
        cv=cv_repeated,
        scoring={
            'accuracy': 'accuracy',
            'recall': 'recall',
            'precision': 'precision',
            'f1': 'f1',
            'roc_auc': 'roc_auc',
            'pr_auc': 'average_precision'
        }
    )

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [116]:
tabla_no_CDR = []
for nombre_modelo in resultados_no_CDR:
    
    for nombre_metrica, metrica in metricas.items():
        
        valores = resultados_no_CDR[nombre_modelo][f'test_{metrica}']

        media = valores.mean()
        desviacion = valores.std()

        resultado_formateado = f'{media:.3f} ± {desviacion:.3f}'
    
        tabla_no_CDR.append({
            'Model': nombre_modelo,
            'Metric': nombre_metrica,
            'Result': resultado_formateado
        })
    

In [118]:
df_no_CDR = pd.DataFrame(tabla_no_CDR) #convertimos la lista en dataframne. 

In [120]:
tabla_pivot_no_CDR = df_no_CDR.pivot(
    index='Metric',
    columns='Model',
    values='Result'
)

In [122]:
tabla_pivot_no_CDR

Model,Logistic,Logistic Balanced,Random Forest,Random Forest Balanced,SVM,SVM balanced
Metric,,,,,,
Accuracy,0.830 ± 0.032,0.590 ± 0.106,0.815 ± 0.050,0.826 ± 0.035,0.837 ± 0.023,0.589 ± 0.106
F1-score,0.008 ± 0.056,0.258 ± 0.164,0.017 ± 0.083,0.000 ± 0.000,0.000 ± 0.000,0.271 ± 0.169
PR-AUC,0.344 ± 0.162,0.340 ± 0.174,0.252 ± 0.148,0.252 ± 0.129,0.355 ± 0.184,0.385 ± 0.189
Precision,0.010 ± 0.070,0.183 ± 0.119,0.027 ± 0.147,0.000 ± 0.000,0.000 ± 0.000,0.193 ± 0.124
ROC-AUC,0.569 ± 0.191,0.546 ± 0.197,0.429 ± 0.169,0.457 ± 0.152,0.619 ± 0.179,0.596 ± 0.198
Recall,0.007 ± 0.047,0.473 ± 0.310,0.013 ± 0.065,0.000 ± 0.000,0.000 ± 0.000,0.500 ± 0.332


### Interpretación

Eliminar CDR **no produjo una disminución clara del rendimiento**. Esto indica que, en este problema concreto, la señal discriminativa no depende principalmente de CDR.

El resultado es coherente con el EDA: 13 de los 14 Converted tenían CDR = 0 en la visita inicial, igual que los 72 Nondemented.


## 7. Ablación 2 — Sin CDR ni MMSE

MMSE también está estrechamente relacionado con el estado cognitivo. El EDA mostró que Converted y Nondemented tenían valores basales muy similares.

Para realizar una prueba más exigente, se eliminan simultáneamente **CDR y MMSE**. El modelo debe trabajar únicamente con sexo, edad, educación, SES, eTIV y nWBV.


In [124]:
X_no_CDR_MMSE = df_baselineSelection2[
    ['M/F', 'Age', 'EDUC', 'SES', 'eTIV', 'nWBV']
]

In [126]:
numeric_features_no_CDR_MMSE = [
    'Age',
    'EDUC',
    'SES',
    'eTIV',
    'nWBV'
]

In [128]:
preprocessor_no_CDR_MMSE = ColumnTransformer(
    transformers=[
        ('num', scaler, numeric_features_no_CDR_MMSE),
        ('cat', encoder, categorical_features)
    ]
)

In [129]:
preprocessor_rf_no_CDR_MMSE = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features_no_CDR_MMSE),
        ('cat', encoder, categorical_features)
    ]
)

In [132]:
pipeline_logistic_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR_MMSE),
    ('classifier', logistic_model)
])

In [133]:
pipeline_logistic_balanced_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR_MMSE),
    ('classifier', logistic_model_balanced)
])

In [134]:
pipeline_svm_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR_MMSE),
    ('classifier', svm_model)
])

In [135]:
pipeline_svm_balanced_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_no_CDR_MMSE),
    ('classifier', svm_model_balanced)
])

In [138]:
pipeline_random_forest_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_rf_no_CDR_MMSE),
    ('classifier', random_forest)
])

In [140]:
pipeline_random_forest_balanced_no_CDR_MMSE = Pipeline(steps=[
    ('preprocessor', preprocessor_rf_no_CDR_MMSE),
    ('classifier', random_forest_balanced)
])

In [142]:
modelos_ML_no_CDR_MMSE = {
    'Logistic': pipeline_logistic_no_CDR_MMSE,
    'Logistic Balanced': pipeline_logistic_balanced_no_CDR_MMSE,
    'Random Forest': pipeline_random_forest_no_CDR_MMSE,
    'Random Forest Balanced': pipeline_random_forest_balanced_no_CDR_MMSE,
    'SVM': pipeline_svm_no_CDR_MMSE,
    'SVM balanced': pipeline_svm_balanced_no_CDR_MMSE 
}

In [143]:
resultados_no_CDR_MMSE = {}

for nombre, modelo in modelos_ML_no_CDR_MMSE.items():
    resultados_no_CDR_MMSE[nombre] = cross_validate(
        modelo,
        X_no_CDR_MMSE,
        y,
        cv=cv_repeated,
        scoring={
            'accuracy': 'accuracy',
            'recall': 'recall',
            'precision': 'precision',
            'f1': 'f1',
            'roc_auc': 'roc_auc',
            'pr_auc': 'average_precision'
        }
    )

C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\marta\anaconda3\Lib\site-packages\sklearn\metrics\_clas

In [148]:
tabla_no_CDR_MMSE = []
for nombre_modelo in resultados_no_CDR_MMSE:
    
    for nombre_metrica, metrica in metricas.items():
        
        valores = resultados_no_CDR_MMSE[nombre_modelo][f'test_{metrica}']

        media = valores.mean()
        desviacion = valores.std()

        resultado_formateado = f'{media:.3f} ± {desviacion:.3f}'
    
        tabla_no_CDR_MMSE.append({
            'Model': nombre_modelo,
            'Metric': nombre_metrica,
            'Result': resultado_formateado
        })

In [150]:
df_no_CDR_MMSE = pd.DataFrame(tabla_no_CDR_MMSE) #convertimos la lista en dataframne. 

In [152]:
tabla_pivot_no_CDR_MMSE = df_no_CDR_MMSE.pivot(
    index='Metric',
    columns='Model',
    values='Result'
)

In [154]:
tabla_pivot_no_CDR_MMSE

Model,Logistic,Logistic Balanced,Random Forest,Random Forest Balanced,SVM,SVM balanced
Metric,,,,,,
Accuracy,0.835 ± 0.032,0.579 ± 0.117,0.807 ± 0.048,0.820 ± 0.039,0.837 ± 0.023,0.604 ± 0.104
F1-score,0.010 ± 0.070,0.288 ± 0.153,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.326 ± 0.155
PR-AUC,0.379 ± 0.175,0.370 ± 0.188,0.242 ± 0.118,0.259 ± 0.131,0.407 ± 0.178,0.417 ± 0.188
Precision,0.020 ± 0.140,0.202 ± 0.115,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.230 ± 0.119
ROC-AUC,0.629 ± 0.178,0.595 ± 0.182,0.471 ± 0.151,0.487 ± 0.169,0.653 ± 0.175,0.646 ± 0.190
Recall,0.007 ± 0.047,0.550 ± 0.306,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.613 ± 0.304


### Interpretación

La eliminación conjunta de CDR y MMSE **no destruyó el rendimiento**. De hecho, algunos resultados medios mejoraron ligeramente, aunque continúan presentando una elevada variabilidad.

Entre las configuraciones evaluadas, el **SVM lineal balanceado sin CDR ni MMSE** mostró uno de los compromisos relativos más interesantes:

- Recall ≈ **0,61**
- Precision ≈ **0,23**
- F1 ≈ **0,33**
- ROC-AUC ≈ **0,65**
- PR-AUC ≈ **0,42**

Esto significa que existe cierta señal multivariable, pero está lejos de ser suficientemente robusta para realizar predicciones individuales fiables.


## 8. ¿De dónde obtiene la señal el modelo?

Como eliminar CDR y MMSE no empeoró el rendimiento, se inspeccionan los coeficientes de los dos modelos lineales balanceados.

Las variables numéricas fueron estandarizadas, por lo que la magnitud de sus coeficientes puede compararse de forma aproximada dentro de cada modelo. Un coeficiente positivo desplaza la predicción hacia `Converted` (1) y uno negativo hacia `Nondemented` (0).

**Importante:** estos coeficientes describen el modelo ajustado a esta pequeña muestra. No representan causalidad ni factores de riesgo poblacionales.


In [156]:
pipeline_logistic_balanced_no_CDR_MMSE.fit(
    X_no_CDR_MMSE,
    y
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'EDUC', 'SES', 'eTIV',
                                                   'nWBV']),
                                                 ('cat', OneHotEncoder(),
                                                  ['M/F'])])),
                ('classifier', LogisticRegression(class_weight='balanced'))])

In [158]:
preprocessor_fitted = pipeline_logistic_balanced_no_CDR_MMSE.named_steps['preprocessor']
logistic_fitted = pipeline_logistic_balanced_no_CDR_MMSE.named_steps['classifier']

In [160]:
feature_names = preprocessor_fitted.get_feature_names_out()
feature_names

array(['num__Age', 'num__EDUC', 'num__SES', 'num__eTIV', 'num__nWBV',
       'cat__M/F_F', 'cat__M/F_M'], dtype=object)

In [162]:
coeficientes = logistic_fitted.coef_.flatten()

In [163]:
interpretacion_logistic = pd.DataFrame({
    'Variable': feature_names,
    'Coeficiente': coeficientes
})

In [164]:
interpretacion_logistic['Importancia'] = interpretacion_logistic['Coeficiente'].abs()

In [168]:
interpretacion_logistic = interpretacion_logistic.sort_values(
    by='Importancia',
    ascending= False
)

In [169]:
interpretacion_logistic

,Variable,Coeficiente,Importancia
2,num__SES,-0.826715,0.826715
1,num__EDUC,-0.531527,0.531527
3,num__eTIV,-0.453956,0.453956
6,cat__M/F_M,0.200435,0.200435
5,cat__M/F_F,-0.199621,0.199621
4,num__nWBV,-0.103733,0.103733
0,num__Age,0.079456,0.079456


In [174]:
pipeline_svm_balanced_no_CDR_MMSE.fit(
    X_no_CDR_MMSE,
    y
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'EDUC', 'SES', 'eTIV',
                                                   'nWBV']),
                                                 ('cat', OneHotEncoder(),
                                                  ['M/F'])])),
                ('classifier',
                 SVC(class_weight='balanced', kernel='linear', probability=True,
                     random_state=42))])

In [175]:
preprocessor_fitted_svm = pipeline_svm_balanced_no_CDR_MMSE.named_steps['preprocessor']
svm_fitted = pipeline_svm_balanced_no_CDR_MMSE.named_steps['classifier']

In [176]:
feature_names_svm = preprocessor_fitted_svm.get_feature_names_out()
feature_names_svm

array(['num__Age', 'num__EDUC', 'num__SES', 'num__eTIV', 'num__nWBV',
       'cat__M/F_F', 'cat__M/F_M'], dtype=object)

In [178]:
coeficientes_svm = svm_fitted.coef_.flatten()

In [181]:
interpretacion_svm = pd.DataFrame({
    'Variable': feature_names_svm,
    'Coeficiente': coeficientes_svm
})

In [182]:
interpretacion_svm['Importancia'] = interpretacion_svm['Coeficiente'].abs()

In [184]:
interpretacion_svm = interpretacion_svm.sort_values(
    by='Importancia',
    ascending=False
)

In [188]:
interpretacion_svm

,Variable,Coeficiente,Importancia
2,num__SES,-1.177560,1.177560
1,num__EDUC,-0.772644,0.772644
3,num__eTIV,-0.536424,0.536424
6,cat__M/F_M,0.401114,0.401114
5,cat__M/F_F,-0.401114,0.401114
4,num__nWBV,-0.298779,0.298779
0,num__Age,0.037913,0.037913


SES y EDUC aparecen entre los coeficientes de mayor magnitud, junto con eTIV. Sin embargo, el EDA mostró una fuerte correlación entre EDUC y SES. Por ello, antes de atribuir importancia a estas variables es necesario comprobar si sus coeficientes son estables cuando una de ellas desaparece.


### Sensibilidad de EDUC y SES


In [190]:
X_no_SES = df_baselineSelection2[
    ['M/F', 'Age', 'EDUC', 'eTIV', 'nWBV']
]

In [191]:
numeric_features_no_SES = [
    'Age',
    'EDUC',
    'eTIV',
    'nWBV'
]

In [193]:
preprocessor_no_SES = ColumnTransformer(
    transformers=[
        ('num', scaler, numeric_features_no_SES),
        ('cat', encoder, categorical_features)
    ]
)

In [195]:
pipeline_logistic_no_SES = Pipeline(steps=[
    ('preprocessor', preprocessor_no_SES),
    ('classifier', logistic_model_balanced)
])

In [197]:
pipeline_logistic_no_SES.fit(
    X_no_SES,
    y
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'EDUC', 'eTIV',
                                                   'nWBV']),
                                                 ('cat', OneHotEncoder(),
                                                  ['M/F'])])),
                ('classifier', LogisticRegression(class_weight='balanced'))])

In [199]:
preprocessor_no_SES_fitted = pipeline_logistic_no_SES.named_steps['preprocessor']
logistic_no_SES_fitted = pipeline_logistic_no_SES.named_steps['classifier']

In [201]:
feature_names_no_SES = preprocessor_no_SES_fitted.get_feature_names_out()
coeficientes_no_SES = logistic_no_SES_fitted.coef_.flatten()

In [202]:
interpretacion_no_SES = pd.DataFrame({
    'Variable': feature_names_no_SES,
    'Coeficiente': coeficientes_no_SES
})

In [203]:
interpretacion_no_SES['Importancia'] = interpretacion_no_SES['Coeficiente'].abs()

interpretacion_no_SES = interpretacion_no_SES.sort_values(
    by='Importancia',
    ascending=False
)

interpretacion_no_SES

,Variable,Coeficiente,Importancia
2,num__eTIV,-0.451181,0.451181
3,num__nWBV,-0.283351,0.283351
5,cat__M/F_M,0.142315,0.142315
4,cat__M/F_F,-0.139993,0.139993
0,num__Age,0.052277,0.052277
1,num__EDUC,-0.036492,0.036492


In [205]:
X_no_EDUC = df_baselineSelection2[
    ['M/F', 'Age', 'SES', 'eTIV', 'nWBV']
]

In [206]:
numeric_features_no_EDUC = [
    'Age',
    'SES',
    'eTIV',
    'nWBV'
]

In [207]:
preprocessor_no_EDUC = ColumnTransformer(
    transformers=[
        ('num', scaler, numeric_features_no_EDUC),
        ('cat', encoder, categorical_features)
    ]
)

In [208]:
pipeline_logistic_no_EDUC = Pipeline(steps=[
    ('preprocessor', preprocessor_no_EDUC),
    ('classifier', logistic_model_balanced)
])

In [212]:
pipeline_logistic_no_EDUC.fit(
    X_no_EDUC,
    y
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Age', 'SES', 'eTIV',
                                                   'nWBV']),
                                                 ('cat', OneHotEncoder(),
                                                  ['M/F'])])),
                ('classifier', LogisticRegression(class_weight='balanced'))])

In [213]:
preprocessor_no_EDUC_fitted = pipeline_logistic_no_EDUC .named_steps['preprocessor']
logistic_no_EDUC_fitted = pipeline_logistic_no_EDUC .named_steps['classifier']

In [214]:
feature_names_no_EDUC = preprocessor_no_EDUC_fitted.get_feature_names_out()
coeficientes_no_EDUC = logistic_no_EDUC_fitted.coef_.flatten()

In [215]:
interpretacion_no_EDUC = pd.DataFrame({
    'Variable': feature_names_no_EDUC,
    'Coeficiente': coeficientes_no_EDUC
})

In [220]:
interpretacion_no_EDUC['Importancia'] = interpretacion_no_EDUC['Coeficiente'].abs()

interpretacion_no_EDUC = interpretacion_no_EDUC.sort_values(
    by='Importancia',
    ascending=False
)

interpretacion_no_EDUC

,Variable,Coeficiente,Importancia
1,num__SES,-0.517438,0.517438
2,num__eTIV,-0.425590,0.425590
0,num__Age,0.120461,0.120461
3,num__nWBV,-0.093831,0.093831
5,cat__M/F_M,0.086928,0.086928
4,cat__M/F_F,-0.086424,0.086424


Al eliminar SES, el coeficiente de EDUC se reduce de forma muy marcada. En cambio, cuando se elimina EDUC, SES conserva una contribución relativamente importante.

Esto demuestra una idea fundamental: **los coeficientes de un modelo multivariable dependen de las demás variables presentes**. Con predictores correlacionados, la importancia puede redistribuirse entre ellos. Por tanto, estos coeficientes deben interpretarse con prudencia y no como evidencia causal.


# Conclusiones del Machine Learning baseline

Los modelos clásicos evaluados muestran una **capacidad limitada para distinguir, utilizando únicamente la visita inicial, a los pacientes Nondemented de aquellos que posteriormente serán clasificados como Converted**.

Los modelos no balanceados alcanzan accuracies cercanas al porcentaje de la clase mayoritaria, pero prácticamente no detectan Converted. El balanceo aumenta la sensibilidad, aunque a costa de numerosos falsos positivos.

Eliminar CDR y posteriormente CDR + MMSE no redujo el rendimiento de forma clara, lo que sugiere que existe alguna señal distribuida entre otras variables basales. Sin embargo, incluso la configuración relativamente más favorable —SVM lineal balanceado sin CDR ni MMSE— presenta discriminación moderada, precision baja y elevada variabilidad.

Por tanto, estos resultados **no permiten afirmar que se haya construido un predictor fiable de Alzheimer o de conversión individual**. El resultado principal es precisamente la dificultad de extraer una señal predictiva robusta de esta pequeña cohorte baseline.


## Limitaciones

1. **Muestra pequeña y muy desbalanceada.** El análisis utiliza 86 pacientes, pero únicamente 14 son Converted. En cada fold de validación hay muy pocos casos positivos.
2. **Elevada variabilidad.** Las desviaciones estándar de las métricas son grandes, lo que indica una fuerte dependencia de cómo se dividen los datos.
3. **Baja precision.** Incluso los modelos balanceados que detectan más Converted generan numerosos falsos positivos.
4. **Discriminación moderada.** Los valores de ROC-AUC y PR-AUC no muestran una separación suficientemente robusta para uso individual.
5. **Sin validación externa.** Todos los experimentos se realizan dentro de OASIS-2.
6. **OASIS-2 no fue diseñado para responder a una predicción presintomática de Alzheimer a cinco años.** El proyecto estudia conversión a demencia dentro de esta cohorte, no establece diagnóstico etiológico de Alzheimer.
7. **Información basal limitada.** No se utilizan imágenes MRI originales, biomarcadores, genética u otras fuentes potencialmente relevantes.
8. **Variables clínicas cercanas al objetivo.** CDR y MMSE pueden introducir circularidad; por ello se realizaron experimentos de ablación.
9. **Colinealidad.** EDUC y SES están relacionados y sus coeficientes no son completamente estables.
10. **Análisis exploratorio.** Se compararon varias configuraciones sobre la misma cohorte sin validación externa ni tuning independiente, por lo que pequeñas diferencias entre modelos no deben sobreinterpretarse.

### Siguiente paso

Si una fotografía del paciente en la primera visita ofrece una señal limitada, la siguiente pregunta es:

**¿La evolución del paciente a lo largo del tiempo contiene información más discriminativa?**

El siguiente notebook aborda esta cuestión mediante un análisis de Machine Learning longitudinal.
